# Pure-space x1 radial spectral diagnostic: Matérn vs alpha=0.7 generalized Cauchy beta grid, nugget comparison

This notebook fits 2023 July pure-space slices with the existing 4x4 ClusterB2 Vecchia geometry.

It is intentionally compact:

- x1 full-grid fits only
- no directional spectra
- three selected calendar days, giving 3 days x 8 GEMS daytime slices = 24 fits per model
- comparison models: Matérn smooth=0.3 plus generalized Cauchy with alpha fixed at 0.7 and beta in {3.0, 3.5, 4.0}
- each covariance shape is fitted twice: nugget fixed at 0 and nugget estimated
- final plot: radial residual spectrum vs fitted expected periodogram, averaged over the selected slices


In [1]:
import gc
import importlib
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.special import gamma, kv

LOCAL_SRC = '/Users/joonwonlee/Documents/GEMS_TCO-1/src'
AMAREL_SRC = '/home/jl2815/tco'
SRC = AMAREL_SRC if Path(AMAREL_SRC).exists() else LOCAL_SRC
if SRC not in sys.path:
    sys.path.insert(0, SRC)

from GEMS_TCO import configuration as config
from GEMS_TCO.data_loader import load_data_dynamic_processed
import GEMS_TCO.kernels_space_base_engine_052126 as kernels_space_base_engine_052126
import GEMS_TCO.kernels_space_aniso_cluster_060326 as kernels_space_aniso_cluster_060326
import GEMS_TCO.kernels_space_aniso_cauchy_cluster_060326 as kernels_space_aniso_cauchy_cluster_060326

importlib.reload(kernels_space_base_engine_052126)
importlib.reload(kernels_space_aniso_cluster_060326)
importlib.reload(kernels_space_aniso_cauchy_cluster_060326)

from GEMS_TCO.kernels_space_aniso_cluster_060326 import (
    ClusterSpaceAnisoTrendVecchiaFit,
    ClusterSpaceAnisoNoNuggetTrendVecchiaFit,
)
from GEMS_TCO.kernels_space_aniso_cauchy_cluster_060326 import (
    ClusterSpaceAnisoCauchyFixedBetaTrendVecchiaFit,
    ClusterSpaceAnisoCauchyFixedBetaNoNuggetTrendVecchiaFit,
    cauchy_phi_init_from_natural,
)

DEVICE = torch.device('cpu')
DTYPE = torch.float64
ROUND_DECIMALS = 4

pd.set_option('display.max_columns', 160)
pd.set_option('display.width', 220)
print('SRC:', SRC)
print('device:', DEVICE)


SRC: /Users/joonwonlee/Documents/GEMS_TCO-1/src
device: cpu


In [2]:
# Experiment config
YEAR = '2023'
MONTH = 7
SELECTED_DAY_NUMBERS = [2, 4, 24]  # Calendar day numbers in July. 3 days x 8 slices/day = 24 time slices.
SLICES_PER_DAY = 8
LAT_RANGE = [-3, 2]
LON_RANGE = [121, 131]

SMOOTH = 0.3
GC_ALPHA_FIXED = 0.7
GC_BETA_GRID = [3.0, 3.5, 4.0]
RESOLUTION_STRIDE = 1
MEAN_DESIGN = 'lat'

CLUSTER_SPEC = {
    'block_shape': (4, 4),
    'n_neighbor_blocks': 2,
    'target_chunk_size': 128,
    'min_target_points': 1,
}


def gc_variant_name(alpha, beta, nugget_mode):
    a = str(float(alpha)).replace('.', 'p')
    b = str(float(beta)).replace('.', 'p')
    return f'gc_a{a}_b{b}_{nugget_mode}'


VARIANTS = {
    'matern_s03_nugget0': {
        'family': 'matern',
        'label': 'Matern s=0.3 nugget0',
        'nugget_mode': 'nugget0',
        'class': ClusterSpaceAnisoNoNuggetTrendVecchiaFit,
        'model': 'ClusterSpaceAnisoMaternS03NoNugget_4x4_B2_exactloc',
        'kernel': 'cluster_space_aniso_matern_s03_nugget0_4x4_b2_exactloc',
        'smooth': 0.3,
        'p_labels': ['sigmasq', 'range_lat', 'range_lon'],
        'init': {'sigmasq': 13.0, 'range_lat': 0.25, 'range_lon': 0.25},
    },
    'matern_s03_nugget_free': {
        'family': 'matern',
        'label': 'Matern s=0.3 nugget free',
        'nugget_mode': 'nugget_free',
        'class': ClusterSpaceAnisoTrendVecchiaFit,
        'model': 'ClusterSpaceAnisoMaternS03NuggetFree_4x4_B2_exactloc',
        'kernel': 'cluster_space_aniso_matern_s03_nuggetfree_4x4_b2_exactloc',
        'smooth': 0.3,
        'p_labels': ['sigmasq', 'range_lat', 'range_lon', 'nugget'],
        'init': {'sigmasq': 13.0, 'range_lat': 0.25, 'range_lon': 0.25, 'nugget': 1.0},
    },
}

for beta in GC_BETA_GRID:
    beta_label = f'{beta:g}'
    common_init = cauchy_phi_init_from_natural(sigmasq=13.0, range_lat=0.25, range_lon=0.25, gc_beta=float(beta))
    name0 = gc_variant_name(GC_ALPHA_FIXED, beta, 'nugget0')
    VARIANTS[name0] = {
        'family': 'cauchy',
        'label': f'GC a={GC_ALPHA_FIXED:g} b={beta_label} nugget0',
        'nugget_mode': 'nugget0',
        'class': ClusterSpaceAnisoCauchyFixedBetaNoNuggetTrendVecchiaFit,
        'model': f'ClusterSpaceAnisoGeneralizedCauchyA07B{beta_label.replace(".", "p")}NoNugget_4x4_B2_exactloc',
        'kernel': f'cluster_space_aniso_gc_a07_b{beta_label.replace(".", "p")}_nugget0_4x4_b2_exactloc',
        'smooth': float(GC_ALPHA_FIXED),
        'gc_alpha': float(GC_ALPHA_FIXED),
        'gc_beta': float(beta),
        'p_labels': ['phi1', 'phi2', 'phi3'],
        'init': dict(common_init),
    }
    name_free = gc_variant_name(GC_ALPHA_FIXED, beta, 'nugget_free')
    VARIANTS[name_free] = {
        'family': 'cauchy',
        'label': f'GC a={GC_ALPHA_FIXED:g} b={beta_label} nugget free',
        'nugget_mode': 'nugget_free',
        'class': ClusterSpaceAnisoCauchyFixedBetaTrendVecchiaFit,
        'model': f'ClusterSpaceAnisoGeneralizedCauchyA07B{beta_label.replace(".", "p")}NuggetFree_4x4_B2_exactloc',
        'kernel': f'cluster_space_aniso_gc_a07_b{beta_label.replace(".", "p")}_nuggetfree_4x4_b2_exactloc',
        'smooth': float(GC_ALPHA_FIXED),
        'gc_alpha': float(GC_ALPHA_FIXED),
        'gc_beta': float(beta),
        'p_labels': ['phi1', 'phi2', 'phi3', 'nugget'],
        'init': {**common_init, 'nugget': 1.0},
    }

LBFGS_LR = 1.0
LBFGS_STEPS_FULL = 8
LBFGS_EVAL = 20
LBFGS_HIST = 10
GRAD_TOL = 1e-5

RUN_FULL = True
RUN_SPECTRUM = True

PURE_SPACE_DIR = Path('/Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/pure_space')
OUT_DIR = PURE_SPACE_DIR / 'log'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PREFIX = 'real_pure_space_spectral_2023_x1_3days_matern_gc_alpha07_beta_grid_nugget_compare_clusterb2_061326'

print('year/month:', YEAR, MONTH)
print('selected days:', SELECTED_DAY_NUMBERS)
print('resolution: x1 only')
print('nugget modes: nugget0 and nugget_free')
print('gc alpha fixed:', GC_ALPHA_FIXED)
print('gc beta grid:', GC_BETA_GRID)
print('variants:', list(VARIANTS))


year/month: 2023 7
selected days: [2, 4, 24]
resolution: x1 only
nugget modes: nugget0 and nugget_free
gc alpha fixed: 0.7
gc beta grid: [3.0, 3.5, 4.0]
variants: ['matern_s03_nugget0', 'matern_s03_nugget_free', 'gc_a0p7_b3p0_nugget0', 'gc_a0p7_b3p0_nugget_free', 'gc_a0p7_b3p5_nugget0', 'gc_a0p7_b3p5_nugget_free', 'gc_a0p7_b4p0_nugget0', 'gc_a0p7_b4p0_nugget_free']


In [3]:
def phys_to_log(init, p_labels):
    return [np.log(init[p]) for p in p_labels]


def backmap(raw, p_labels, variant):
    spec = VARIANTS[variant]
    est_raw = {p: float(np.exp(raw[i])) for i, p in enumerate(p_labels)}
    if {'phi1', 'phi2', 'phi3'}.issubset(est_raw):
        phi1 = est_raw['phi1']
        phi2 = est_raw['phi2']
        phi3 = est_raw['phi3']
        range_lon = 1.0 / phi2
        range_lat = 1.0 / (phi2 * np.sqrt(phi3))
        sigmasq = phi1 / phi2
        return {
            **est_raw,
            'sigmasq': float(sigmasq),
            'range_lat': float(range_lat),
            'range_lon': float(range_lon),
            'range': float(range_lon),
            'nugget': float(est_raw.get('nugget', 0.0)),
            'gc_alpha': float(spec['gc_alpha']),
            'gc_beta': float(spec['gc_beta']),
        }
    est = {p: float(np.exp(raw[i])) for i, p in enumerate(p_labels)}
    if 'range_lat' in est and 'range_lon' in est:
        est['range'] = est['range_lon']
    elif 'range' in est:
        est['range_lat'] = est['range']
        est['range_lon'] = est['range']
    if 'nugget' not in est:
        est['nugget'] = 0.0
    est['gc_alpha'] = np.nan
    est['gc_beta'] = np.nan
    return est


def make_full_params(variant):
    spec = VARIANTS[variant]
    return [torch.tensor([v], device=DEVICE, dtype=DTYPE, requires_grad=True) for v in phys_to_log(spec['init'], spec['p_labels'])]


def count_valid(input_map):
    return int(sum(((~torch.isnan(v[:, 2])) & (~torch.isnan(v[:, 0])) & (~torch.isnan(v[:, 1]))).sum().item() for v in input_map.values()))


def space_diag(model):
    if hasattr(model, 'cluster_summary'):
        return dict(model.cluster_summary())
    groups = getattr(model, 'Batched_Groups', []) or []
    if not groups:
        return {'n_batches': 0, 'n_tails': 0, 'mean_m': 0.0, 'max_m': 0, 'largest_batch_n': 0}
    ns = np.asarray([int(g['target_idx'].shape[0]) for g in groups], dtype=np.int64)
    ms = np.asarray([int(g['max_m']) for g in groups], dtype=np.int64)
    return {
        'n_batches': int(len(groups)),
        'n_tails': int(ns.sum()),
        'mean_m': float(ms.mean()),
        'max_m': int(ms.max()),
        'largest_batch_n': int(ns.max()),
    }


def round_df(df, digits=ROUND_DECIMALS):
    out = df.copy()
    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].round(digits)
    return out


def empty_cache():
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()


def resolution_label(stride):
    return f'x{int(stride)}'


In [4]:
loader = load_data_dynamic_processed(config.mac_data_load_path)
df_map, _, _, raw_monthly_mean = loader.load_maxmin_ordered_data_bymonthyear(
    lat_lon_resolution=[1, 1],
    mm_cond_number=10,
    years_=[YEAR],
    months_=[MONTH],
    lat_range=LAT_RANGE,
    lon_range=LON_RANGE,
    is_whittle=True,
)
month_keys = sorted(df_map.keys())
if len(month_keys) < SLICES_PER_DAY:
    raise ValueError(f'Only {len(month_keys)} slices found for {YEAR}-{MONTH:02d}')
if len(month_keys) % SLICES_PER_DAY != 0:
    print(f'Warning: {len(month_keys)} slices is not divisible by SLICES_PER_DAY={SLICES_PER_DAY}; trailing slices are ignored by day indexing.')

n_complete_days = len(month_keys) // SLICES_PER_DAY
selected_positions = []
selected_rows = []
for day_num in SELECTED_DAY_NUMBERS:
    if day_num < 1 or day_num > n_complete_days:
        raise ValueError(f'Selected day {day_num} outside available complete-day range 1..{n_complete_days}')
    start = (int(day_num) - 1) * SLICES_PER_DAY
    for slice_in_day in range(SLICES_PER_DAY):
        pos = start + slice_in_day
        key = month_keys[pos]
        selected_positions.append(pos)
        selected_rows.append({
            'calendar_day': int(day_num),
            'slice_in_day': int(slice_in_day),
            'time_position': int(pos),
            'time_key': key,
        })
selected_time_df = pd.DataFrame(selected_rows)
selected_time_path = OUT_DIR / f'{OUT_PREFIX}_selected_time_slices.csv'
selected_time_df.to_csv(selected_time_path, index=False)

first_df = df_map[month_keys[0]].reset_index(drop=True)
grid_order = (
    first_df
    .assign(_orig_idx=np.arange(len(first_df)))
    .sort_values(['Longitude', 'Latitude', '_orig_idx'], kind='mergesort')['_orig_idx']
    .to_numpy(dtype=np.int64)
)
grid_coords_full = first_df.iloc[grid_order][['Latitude', 'Longitude']].to_numpy(dtype=np.float64)

_lat_key = np.round(grid_coords_full[:, 0], 10)
_lon_key = np.round(grid_coords_full[:, 1], 10)
lat_vals = np.sort(np.unique(_lat_key))
lon_vals = np.sort(np.unique(_lon_key))
lat_to_row = {float(v): i for i, v in enumerate(lat_vals)}
lon_to_col = {float(v): i for i, v in enumerate(lon_vals)}
local_to_row = np.asarray([lat_to_row[float(v)] for v in _lat_key], dtype=np.int64)
local_to_col = np.asarray([lon_to_col[float(v)] for v in _lon_key], dtype=np.int64)
N_LAT, N_LON = len(lat_vals), len(lon_vals)
LAT_STEP = float(np.median(np.diff(lat_vals))) if N_LAT > 1 else 1.0
LON_STEP = float(np.median(np.diff(lon_vals))) if N_LON > 1 else 1.0

ANALYSIS_MONTHLY_MEAN = 0.0
print('raw monthly_mean:', round(raw_monthly_mean, 4), '(not subtracted; intercept handles the level)')
print('available complete days:', n_complete_days)
print('selected time slices:', len(selected_time_df))
print('selected time path:', selected_time_path)
print('first/last selected key:', selected_time_df['time_key'].iloc[0], selected_time_df['time_key'].iloc[-1])
print('full grid:', grid_coords_full.shape)
print('unique lat/lon:', N_LAT, N_LON)
print('lat/lon step:', LAT_STEP, LON_STEP)
display(selected_time_df)


--- Global Monthly Mean for 2023-7: 249.7131 ---
raw monthly_mean: 249.7131 (not subtracted; intercept handles the level)
available complete days: 31
selected time slices: 24
selected time path: /Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/pure_space/log/real_pure_space_spectral_2023_x1_3days_matern_gc_alpha07_beta_grid_nugget_compare_clusterb2_061326_selected_time_slices.csv
first/last selected key: 2023_07_y23m07day02_hm00:53 2023_07_y23m07day24_hm07:48
full grid: (18126, 2)
unique lat/lon: 114 159
lat/lon step: 0.04400000000000004 0.06300000000000239


,calendar_day,slice_in_day,time_position,time_key
0,2,0,8,2023_07_y23m07day02_hm00:53
1,2,1,9,2023_07_y23m07day02_hm01:53
2,2,2,10,2023_07_y23m07day02_hm02:48
3,2,3,11,2023_07_y23m07day02_hm03:48
4,2,4,12,2023_07_y23m07day02_hm04:48
5,2,5,13,2023_07_y23m07day02_hm05:48
6,2,6,14,2023_07_y23m07day02_hm06:48
7,2,7,15,2023_07_y23m07day02_hm07:48
8,4,0,24,2023_07_y23m07day04_hm00:53
9,4,1,25,2023_07_y23m07day04_hm01:53


In [5]:
def load_time_map(time_position):
    key = month_keys[int(time_position)]
    time_df_map = {key: df_map[key]}
    time_map, _ = loader.load_working_data(
        time_df_map,
        monthly_mean=ANALYSIS_MONTHLY_MEAN,
        idx_for_datamap=[0, 1],
        ord_mm=grid_order,
        dtype=DTYPE,
        keep_ori=True,
    )
    return {k: v.to(DEVICE) for k, v in time_map.items()}, key


def thin_time_map(time_map, stride=RESOLUTION_STRIDE):
    stride = int(stride)
    if stride <= 0:
        raise ValueError(f'stride must be positive, got {stride}')
    keep = (local_to_row % stride == 0) & (local_to_col % stride == 0)
    thin_idx = np.flatnonzero(keep).astype(np.int64)
    thin_map = {k: v[thin_idx].contiguous() for k, v in time_map.items()}
    thin_grid = np.ascontiguousarray(grid_coords_full[thin_idx])
    return thin_map, thin_grid, thin_idx


def build_model(variant, input_map, grid_coords):
    spec = VARIANTS[variant]
    kwargs = dict(
        smooth=float(spec.get('smooth', SMOOTH)),
        input_map=input_map,
        grid_coords=np.ascontiguousarray(grid_coords.astype(np.float64)),
        block_shape=CLUSTER_SPEC['block_shape'],
        n_neighbor_blocks=CLUSTER_SPEC['n_neighbor_blocks'],
        target_chunk_size=CLUSTER_SPEC['target_chunk_size'],
        min_target_points=CLUSTER_SPEC['min_target_points'],
        mean_design=MEAN_DESIGN,
    )
    if spec['family'] == 'cauchy':
        kwargs.update(gc_alpha=float(spec['gc_alpha']), gc_beta=float(spec['gc_beta']))
    return spec['class'](**kwargs)


def make_base_row(variant, calendar_day, slice_in_day, time_position, time_key, n_grid, n_valid, pre_s, fit_s, loss, fit_iter, est, diag):
    spec = VARIANTS[variant]
    row = {
        'date_group': f'{YEAR}{MONTH:02d}_days_' + '_'.join(str(d) for d in SELECTED_DAY_NUMBERS),
        'year': int(YEAR),
        'month': int(MONTH),
        'calendar_day': int(calendar_day),
        'slice_in_day': int(slice_in_day),
        'time_position': int(time_position),
        'time_key': str(time_key),
        'resolution_stride': int(RESOLUTION_STRIDE),
        'resolution_label': resolution_label(RESOLUTION_STRIDE),
        'variant': variant,
        'model_family': spec['family'],
        'model_label': spec['label'],
        'nugget_mode': spec.get('nugget_mode', 'unknown'),
        'smooth': float(spec.get('smooth', np.nan)),
        'gc_alpha': float(spec.get('gc_alpha', np.nan)),
        'gc_beta': float(spec.get('gc_beta', np.nan)),
        'mean_design': MEAN_DESIGN,
        'fit_type': 'full_x1',
        'model': spec['model'],
        'kernel': spec['kernel'],
        'coord_mode': 'x1 full grid; 4x4 cluster max-min ordering on cluster centroids; condition on nearest 2 previous cluster blocks; anisotropic covariance on Source_Latitude/Source_Longitude',
        'loss': float(loss),
        'fit_iter_raw': int(fit_iter),
        'fit_steps_reported': int(fit_iter) + 1,
        'precompute_s': float(pre_s),
        'fit_s': float(fit_s),
        'total_s': float(pre_s + fit_s),
        'n_grid': int(n_grid),
        'n_valid': int(n_valid),
        'valid_fraction': float(n_valid / n_grid) if n_grid else np.nan,
        'est_sigmasq': float(est['sigmasq']),
        'est_range': float(est['range_lon']),
        'est_range_lat': float(est['range_lat']),
        'est_range_lon': float(est['range_lon']),
        'est_nugget': float(est.get('nugget', 0.0)),
        'est_gc_alpha': float(est.get('gc_alpha', np.nan)),
        'est_gc_beta': float(est.get('gc_beta', np.nan)),
        'est_phi1': float(est.get('phi1', np.nan)),
        'est_phi2': float(est.get('phi2', np.nan)),
        'est_phi3': float(est.get('phi3', np.nan)),
        **diag,
        'cluster_block_shape': f"{CLUSTER_SPEC['block_shape'][0]}x{CLUSTER_SPEC['block_shape'][1]}",
        'cluster_neighbor_blocks': CLUSTER_SPEC['n_neighbor_blocks'],
        'total_conditioning_nominal': int(CLUSTER_SPEC['n_neighbor_blocks'] * np.prod(CLUSTER_SPEC['block_shape'])),
    }
    return row


def fit_full_variant(variant, calendar_day, slice_in_day, time_position):
    time_map, time_key = load_time_map(time_position)
    thin_map, thin_grid, _ = thin_time_map(time_map, RESOLUTION_STRIDE)
    n_grid = int(thin_grid.shape[0])
    n_valid = count_valid(thin_map)
    spec = VARIANTS[variant]
    print('\n' + '=' * 100)
    print(f"{variant} | {spec['label']} | day={calendar_day} slice={slice_in_day} | {time_key} | x1 | n_grid={n_grid:,} | n_valid={n_valid:,}")

    model = build_model(variant, thin_map, thin_grid)
    t_pre = time.time()
    model.precompute_conditioning_sets()
    pre_s = time.time() - t_pre
    diag = space_diag(model)

    params = make_full_params(variant)
    opt = model.set_optimizer(params, lr=LBFGS_LR, max_iter=LBFGS_EVAL, max_eval=LBFGS_EVAL, history_size=LBFGS_HIST)
    t_fit = time.time()
    out, fit_iter = model.fit_vecc_lbfgs(params, opt, max_steps=LBFGS_STEPS_FULL, grad_tol=GRAD_TOL)
    fit_s = time.time() - t_fit
    p_labels = spec['p_labels']
    est = backmap(out[:len(p_labels)], p_labels, variant)
    row = make_base_row(variant, calendar_day, slice_in_day, time_position, time_key, n_grid, n_valid, pre_s, fit_s, out[-1], fit_iter, est, diag)
    print('RESULT:', {k: round(v, 4) if isinstance(v, float) else v for k, v in row.items() if k in ['variant','loss','est_sigmasq','est_range_lat','est_range_lon','est_gc_alpha','est_gc_beta','total_s']})

    del model, params, opt, time_map, thin_map
    empty_cache()
    return row


In [ ]:
fit_rows = []
if RUN_FULL:
    selected_records = selected_time_df.to_dict('records')
    for variant in VARIANTS:
        for rec in selected_records:
            row = fit_full_variant(
                variant,
                rec['calendar_day'],
                rec['slice_in_day'],
                rec['time_position'],
            )
            fit_rows.append(row)
            tmp = round_df(pd.DataFrame(fit_rows))
            tmp.to_csv(OUT_DIR / f'{OUT_PREFIX}_full.csv', index=False, float_format=f'%.{ROUND_DECIMALS}f')

fit_df = pd.DataFrame(fit_rows)
full_path = OUT_DIR / f'{OUT_PREFIX}_full.csv'
round_df(fit_df).to_csv(full_path, index=False, float_format=f'%.{ROUND_DECIMALS}f')
print('Saved full fits:', full_path)

display_cols = [
    'variant', 'model_label', 'calendar_day', 'slice_in_day', 'loss',
    'est_sigmasq', 'est_range_lat', 'est_range_lon', 'est_nugget', 'est_gc_alpha', 'est_gc_beta',
    'n_grid', 'n_valid', 'total_s',
]
display(round_df(fit_df[display_cols]))

summary_cols = {
    'loss': ['mean', 'median'],
    'est_sigmasq': ['mean', 'median'],
    'est_range_lat': ['mean', 'median'],
    'est_range_lon': ['mean', 'median'],
    'total_s': ['mean', 'sum'],
}
fit_summary = fit_df.groupby(['variant', 'model_label'], observed=False).agg(summary_cols)
fit_summary.columns = ['_'.join(c).strip() for c in fit_summary.columns]
fit_summary = fit_summary.reset_index()
summary_path = OUT_DIR / f'{OUT_PREFIX}_fit_summary_by_variant.csv'
round_df(fit_summary).to_csv(summary_path, index=False, float_format=f'%.{ROUND_DECIMALS}f')
print('Saved fit summary:', summary_path)
display(round_df(fit_summary))



matern_s03_nugget0 | Matern s=0.3 nugget0 | day=2 slice=0 | 2023_07_y23m07day02_hm00:53 | x1 | n_grid=18,126 | n_valid=17,747
Pre-computing ClusterSpaceVecchia [block=4x4, B=2]... Done in 0.1s. clusters=1160, max_points/block=16, target_blocks=1158, target_points=17747, m mean/med/max=30.3/32/32, target med/max=16/16
--- Starting Pure-Space Vecchia L-BFGS ---
--- Step 1/8 / Loss: 1.360785 / Max Grad: 2.86e-06 ---
Converged: max_grad 2.86e-06 < 1.00e-05
Final Pure-Space Params: {'sigmasq': 10.667389771950504, 'range_lat': 0.14336132571044952, 'range_lon': 0.16743269762477428, 'range': 0.16743269762477428, 'nugget': 0.0}
RESULT: {'variant': 'matern_s03_nugget0', 'loss': 1.3608, 'total_s': 1.2621, 'est_sigmasq': 10.6674, 'est_range_lat': 0.1434, 'est_range_lon': 0.1674, 'est_gc_alpha': nan, 'est_gc_beta': nan}

matern_s03_nugget0 | Matern s=0.3 nugget0 | day=2 slice=1 | 2023_07_y23m07day02_hm01:53 | x1 | n_grid=18,126 | n_valid=17,680
Pre-computing ClusterSpaceVecchia [block=4x4, B=2]...

In [ ]:
RADIAL_N_BINS = 70
RADIAL_QMAX = 0.985
APPLY_HANN = False
EPS = 1e-12

print('spectral full grid:', (N_LAT, N_LON), 'lat/lon step:', LAT_STEP, LON_STEP)


def trend_design(lat, lon, lat_center=None, lon_center=None):
    lat_center = np.nanmean(lat) if lat_center is None else float(lat_center)
    lon_center = np.nanmean(lon) if lon_center is None else float(lon_center)
    lat_c = lat - lat_center
    lon_c = lon - lon_center
    if MEAN_DESIGN in ('base', 'lat'):
        return np.column_stack([np.ones(len(lat)), lat_c])
    return np.column_stack([np.ones(len(lat)), lat_c, lon_c])


def detrended_residual_grid(time_position, stride=RESOLUTION_STRIDE):
    time_map, time_key = load_time_map(time_position)
    thin_map, thin_grid, thin_idx = thin_time_map(time_map, stride)
    arr = next(iter(thin_map.values())).detach().cpu().numpy()
    y = arr[:, 2].astype(float)
    lat = arr[:, 0].astype(float)
    lon = arr[:, 1].astype(float)
    valid = np.isfinite(y) & np.isfinite(lat) & np.isfinite(lon)
    if valid.sum() < 4:
        raise ValueError(f'Not enough valid points for time_position={time_position}, stride={stride}')

    lat_center = np.nanmean(lat[valid])
    lon_center = np.nanmean(lon[valid])
    X = trend_design(lat[valid], lon[valid], lat_center=lat_center, lon_center=lon_center)
    beta, *_ = np.linalg.lstsq(X, y[valid], rcond=None)
    X_all = trend_design(lat, lon, lat_center=lat_center, lon_center=lon_center)
    resid = y - X_all @ beta

    rows_full = local_to_row[thin_idx]
    cols_full = local_to_col[thin_idx]
    row_keep = np.sort(np.unique(rows_full))
    col_keep = np.sort(np.unique(cols_full))
    row_pos = {int(r): i for i, r in enumerate(row_keep)}
    col_pos = {int(c): i for i, c in enumerate(col_keep)}

    grid = np.full((len(row_keep), len(col_keep)), np.nan, dtype=float)
    mask = np.zeros_like(grid, dtype=float)
    rr = np.asarray([row_pos[int(r)] for r in rows_full[valid]], dtype=np.int64)
    cc = np.asarray([col_pos[int(c)] for c in cols_full[valid]], dtype=np.int64)
    grid[rr, cc] = resid[valid]
    mask[rr, cc] = 1.0

    lat_axis = lat_vals[row_keep].astype(float)
    lon_axis = lon_vals[col_keep].astype(float)
    return grid, mask, time_key, int(valid.sum()), lat_axis, lon_axis


def masked_periodogram(grid, mask):
    obs = mask > 0
    if not np.any(obs):
        raise ValueError('No observed cells in spectral grid.')
    z = np.zeros_like(grid, dtype=float)
    z[obs] = grid[obs]
    win = np.outer(np.hanning(z.shape[0]), np.hanning(z.shape[1])) if APPLY_HANN else np.ones_like(z)
    zw = z * win
    norm = np.sum((mask * win) ** 2)
    norm = norm if norm > EPS else 1.0
    return np.abs(np.fft.fftshift(np.fft.fft2(zw))) ** 2 / norm


def matern_covariance_lag(sigmasq, range_lat, range_lon, nugget, smooth, lag_lat, lag_lon):
    lag_lat = np.asarray(lag_lat, dtype=float)
    lag_lon = np.asarray(lag_lon, dtype=float)
    r = np.sqrt((lag_lat / max(float(range_lat), EPS)) ** 2 + (lag_lon / max(float(range_lon), EPS)) ** 2)
    nu = float(smooth)
    if np.isclose(nu, 0.5):
        corr = np.exp(-r)
    elif np.isclose(nu, 1.5):
        corr = (1.0 + r) * np.exp(-r)
    else:
        corr = np.ones_like(r, dtype=float)
        positive = r > EPS
        if np.any(positive):
            scaled = np.sqrt(2.0 * nu) * r[positive]
            corr[positive] = (2.0 ** (1.0 - nu) / gamma(nu)) * (scaled ** nu) * kv(nu, scaled)
        corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
    cov = float(sigmasq) * corr
    zero_lag = (np.abs(lag_lat) < EPS) & (np.abs(lag_lon) < EPS)
    return cov + max(float(nugget), 0.0) * zero_lag


def cauchy_covariance_lag(sigmasq, range_lat, range_lon, nugget, gc_alpha, gc_beta, lag_lat, lag_lon):
    lag_lat = np.asarray(lag_lat, dtype=float)
    lag_lon = np.asarray(lag_lon, dtype=float)
    d = np.sqrt((lag_lat / max(float(range_lat), EPS)) ** 2 + (lag_lon / max(float(range_lon), EPS)) ** 2)
    alpha = max(float(gc_alpha), EPS)
    beta = max(float(gc_beta), EPS)
    corr = np.power(1.0 + np.power(d, alpha), -beta / alpha)
    cov = float(sigmasq) * corr
    zero_lag = (np.abs(lag_lat) < EPS) & (np.abs(lag_lon) < EPS)
    return cov + max(float(nugget), 0.0) * zero_lag


def covariance_lag(row, lag_lat, lag_lon):
    if str(row.model_family) == 'matern':
        return matern_covariance_lag(row.est_sigmasq, row.est_range_lat, row.est_range_lon, row.est_nugget, row.smooth, lag_lat, lag_lon)
    return cauchy_covariance_lag(row.est_sigmasq, row.est_range_lat, row.est_range_lon, row.est_nugget, row.est_gc_alpha, row.est_gc_beta, lag_lat, lag_lon)


def mask_window_autocorrelation(mask):
    mask = np.asarray(mask, dtype=float)
    win = np.outer(np.hanning(mask.shape[0]), np.hanning(mask.shape[1])) if APPLY_HANN else np.ones_like(mask)
    g = mask * win
    H = float(np.sum(g ** 2))
    if H <= EPS:
        raise ValueError('Taper/mask normalization is zero.')
    n1, n2 = g.shape
    G = np.fft.fft2(g, s=(2 * n1 - 1, 2 * n2 - 1))
    ac = np.fft.fftshift(np.fft.ifft2(G * np.conj(G)).real) / H
    return ac, H


def _autocorr_for_lags(ac, lag1, lag2, n1, n2):
    out = np.zeros_like(lag1, dtype=float)
    valid = (np.abs(lag1) <= n1 - 1) & (np.abs(lag2) <= n2 - 1)
    if np.any(valid):
        out[valid] = ac[(n1 - 1 + lag1[valid]).astype(int), (n2 - 1 + lag2[valid]).astype(int)]
    return out


def axis_step(axis_vals):
    axis_vals = np.asarray(axis_vals, dtype=float)
    return float(np.median(np.diff(axis_vals))) if len(axis_vals) > 1 else 1.0


def frequency_grid_for_axes(lat_axis, lon_axis):
    dlat = axis_step(lat_axis)
    dlon = axis_step(lon_axis)
    fy = np.fft.fftshift(np.fft.fftfreq(len(lat_axis), d=dlat))
    fx = np.fft.fftshift(np.fft.fftfreq(len(lon_axis), d=dlon))
    omega_y = 2.0 * np.pi * fy
    omega_x = 2.0 * np.pi * fx
    OX, OY = np.meshgrid(omega_x, omega_y)
    k = np.sqrt(OX ** 2 + OY ** 2)
    return k, OX ** 2 + OY ** 2, omega_y, omega_x


K_FULL_RADIAL, OMEGA2_FULL, OMEGA_LAT_FULL, OMEGA_LON_FULL = frequency_grid_for_axes(lat_vals, lon_vals)
_positive_k = K_FULL_RADIAL[np.isfinite(K_FULL_RADIAL) & (K_FULL_RADIAL > 0)]
K_MAX_FULL = float(np.quantile(_positive_k, RADIAL_QMAX))
RADIAL_BINS = np.linspace(0.0, K_MAX_FULL, RADIAL_N_BINS + 1)
FULL_K_PLOT_MAX = K_MAX_FULL


def radial_average(surface, k_radial, bins=RADIAL_BINS, k_max=K_MAX_FULL):
    vals = np.asarray(surface, dtype=float).ravel()
    kk = np.asarray(k_radial, dtype=float).ravel()
    good = np.isfinite(vals) & np.isfinite(kk) & (kk > 0) & (kk <= k_max)
    bin_idx = np.digitize(kk[good], bins) - 1
    rows = []
    vg = vals[good]
    kg = kk[good]
    for b in range(len(bins) - 1):
        m = bin_idx == b
        if not np.any(m):
            continue
        rows.append({
            'k_bin': b,
            'k_mid': 0.5 * (bins[b] + bins[b + 1]),
            'k_mean': float(np.nanmean(kg[m])),
            'spectrum': float(np.nanmean(vg[m])),
            'n_freq': int(m.sum()),
        })
    return pd.DataFrame(rows)


def fft_convolve_same(values, kernel):
    values = np.asarray(values, dtype=float)
    kernel = np.asarray(kernel, dtype=float)
    out_shape = (values.shape[0] + kernel.shape[0] - 1, values.shape[1] + kernel.shape[1] - 1)
    conv = np.fft.ifft2(np.fft.fft2(values, s=out_shape) * np.fft.fft2(kernel, s=out_shape)).real
    start0 = kernel.shape[0] // 2
    start1 = kernel.shape[1] // 2
    return conv[start0:start0 + values.shape[0], start1:start1 + values.shape[1]]


def covariance_lag_kernel(row, n1, n2, dlat, dlon):
    lag1_vals = np.arange(-(n1 - 1), n1, dtype=int)
    lag2_vals = np.arange(-(n2 - 1), n2, dtype=int)
    lag1, lag2 = np.meshgrid(lag1_vals, lag2_vals, indexing='ij')
    return covariance_lag(row, lag1 * dlat, lag2 * dlon)


def expected_periodogram_dw_style(row, mask, lat_axis, lon_axis, project_mean=True):
    n1, n2 = mask.shape
    ac, _ = mask_window_autocorrelation(mask)
    dlat = axis_step(lat_axis)
    dlon = axis_step(lon_axis)
    u1, u2 = np.meshgrid(np.arange(n1, dtype=int), np.arange(n2, dtype=int), indexing='ij')
    tilde_cov = np.zeros((n1, n2), dtype=float)
    for shift1 in (0, -n1):
        for shift2 in (0, -n2):
            lag1 = u1 + shift1
            lag2 = u2 + shift2
            ac_lag = _autocorr_for_lags(ac, lag1, lag2, n1, n2)
            if not np.any(ac_lag):
                continue
            tilde_cov += covariance_lag(row, lag1 * dlat, lag2 * dlon) * ac_lag
    expected_no_projection = np.fft.fftshift(np.fft.fft2(tilde_cov).real)
    if not project_mean:
        return np.maximum(expected_no_projection, EPS)

    obs = mask > 0
    if obs.sum() <= 2:
        return np.maximum(expected_no_projection, EPS)
    win = np.outer(np.hanning(n1), np.hanning(n2)) if APPLY_HANN else np.ones((n1, n2), dtype=float)
    g = mask * win
    H = float(np.sum(g ** 2))
    lat_grid = np.repeat(np.asarray(lat_axis, dtype=float)[:, None], n2, axis=1)
    lon_grid = np.repeat(np.asarray(lon_axis, dtype=float)[None, :], n1, axis=0)
    X_flat = trend_design(lat_grid.ravel(), lon_grid.ravel(), lat_center=float(np.nanmean(lat_grid[obs])), lon_center=float(np.nanmean(lon_grid[obs])))
    X = X_flat.reshape(n1, n2, X_flat.shape[1])
    U = X * mask[..., None]
    XtX = np.einsum('ijp,ijq->pq', U, X)
    B = np.linalg.pinv(XtX)

    kernel = covariance_lag_kernel(row, n1, n2, dlat, dlon)
    CU = np.empty_like(U, dtype=float)
    for p_idx in range(U.shape[2]):
        CU[..., p_idx] = fft_convolve_same(U[..., p_idx], kernel)
    S = np.einsum('ijp,ijq->pq', U, CU)

    h_fft = []
    c_fft = []
    for p_idx in range(U.shape[2]):
        h_fft.append(np.fft.fftshift(np.fft.fft2(g * X[..., p_idx])))
        c_fft.append(np.fft.fftshift(np.fft.fft2(g * CU[..., p_idx])))
    h_fft = np.stack(h_fft, axis=-1)
    c_fft = np.stack(c_fft, axis=-1)

    A_raw = expected_no_projection * H
    cross = np.einsum('...p,pq,...q->...', c_fft, B, np.conj(h_fft))
    mean_quad = np.einsum('...p,pq,...q->...', h_fft, B @ S @ B, np.conj(h_fft))
    expected_projected = (A_raw - 2.0 * np.real(cross) + np.real(mean_quad)) / H
    return np.maximum(expected_projected, EPS)


In [ ]:
if RUN_SPECTRUM:
    if 'fit_df' not in globals() or fit_df.empty:
        fit_df = pd.read_csv(OUT_DIR / f'{OUT_PREFIX}_full.csv')

    data_cache = {}
    spectral_rows = []
    for r in fit_df.itertuples(index=False):
        cache_key = int(r.time_position)
        if cache_key not in data_cache:
            grid, mask, time_key, n_valid_spectrum, lat_axis, lon_axis = detrended_residual_grid(cache_key, RESOLUTION_STRIDE)
            data_p = masked_periodogram(grid, mask)
            k_data, _, _, _ = frequency_grid_for_axes(lat_axis, lon_axis)
            data_rad = radial_average(data_p, k_data).rename(columns={'spectrum': 'data_spectrum'})
            data_cache[cache_key] = (mask, time_key, n_valid_spectrum, lat_axis, lon_axis, k_data, data_rad)
        mask, time_key, n_valid_spectrum, lat_axis, lon_axis, k_data, data_rad = data_cache[cache_key]

        theory_expected_p = expected_periodogram_dw_style(r, mask, lat_axis, lon_axis)
        theory_expected_rad = radial_average(theory_expected_p, k_data).rename(columns={'spectrum': 'theory_spectrum_expected'})
        merged = theory_expected_rad.merge(data_rad[['k_bin', 'data_spectrum', 'n_freq']], on='k_bin', how='left', suffixes=('', '_data'))
        data_k_max = float(data_rad['k_mid'].max()) if not data_rad.empty else np.nan

        for m in merged.itertuples(index=False):
            spectral_rows.append({
                'year': int(r.year),
                'month': int(r.month),
                'calendar_day': int(r.calendar_day),
                'slice_in_day': int(r.slice_in_day),
                'time_position': int(r.time_position),
                'time_key': time_key,
                'resolution_stride': int(r.resolution_stride),
                'resolution_label': r.resolution_label,
                'variant': r.variant,
                'model_family': r.model_family,
                'model_label': r.model_label,
                'smooth': float(r.smooth),
                'gc_alpha': float(r.gc_alpha) if pd.notna(r.gc_alpha) else np.nan,
                'gc_beta': float(r.gc_beta) if pd.notna(r.gc_beta) else np.nan,
                'n_valid_spectrum': n_valid_spectrum,
                'est_sigmasq': float(r.est_sigmasq),
                'est_range': float(r.est_range_lon),
                'est_range_lat': float(r.est_range_lat),
                'est_range_lon': float(r.est_range_lon),
                'est_nugget': 0.0,
                'est_gc_alpha': float(r.est_gc_alpha) if pd.notna(r.est_gc_alpha) else np.nan,
                'est_gc_beta': float(r.est_gc_beta) if pd.notna(r.est_gc_beta) else np.nan,
                'k_bin': int(m.k_bin),
                'k_mid': float(m.k_mid),
                'k_mean': float(m.k_mean),
                'n_freq': int(getattr(m, 'n_freq_data')) if pd.notna(getattr(m, 'n_freq_data', np.nan)) else 0,
                'data_k_max': data_k_max,
                'data_spectrum': float(m.data_spectrum) if pd.notna(m.data_spectrum) else np.nan,
                'theory_spectrum_expected': float(m.theory_spectrum_expected) if pd.notna(m.theory_spectrum_expected) else np.nan,
            })

    spectral_df = pd.DataFrame(spectral_rows)
    spectral_path = OUT_DIR / f'{OUT_PREFIX}_radial_spectrum.csv'
    spectral_df.to_csv(spectral_path, index=False)
    print('Saved radial spectrum:', spectral_path)
    display(round_df(spectral_df.head(12)))


In [ ]:
# Average x1 radial diagnostic plot: one panel per model.
if 'spectral_df' not in globals():
    spectral_df = pd.read_csv(OUT_DIR / f'{OUT_PREFIX}_radial_spectrum.csv')
if 'fit_df' not in globals():
    fit_df = pd.read_csv(OUT_DIR / f'{OUT_PREFIX}_full.csv')

plot_spec = spectral_df.copy()
plot_spec = plot_spec.replace([np.inf, -np.inf], np.nan)

avg_spec = (
    plot_spec
    .groupby(['variant', 'model_label', 'k_bin'], observed=False)
    .agg(
        k_mid=('k_mid', 'mean'),
        data_spectrum=('data_spectrum', 'mean'),
        theory_spectrum_expected=('theory_spectrum_expected', 'mean'),
        n_slices=('time_position', 'nunique'),
        data_k_max=('data_k_max', 'mean'),
    )
    .reset_index()
)
avg_spec['ratio_I_over_EI'] = avg_spec['data_spectrum'] / avg_spec['theory_spectrum_expected']
avg_path = OUT_DIR / f'{OUT_PREFIX}_radial_spectrum_avg_by_variant.csv'
avg_spec.to_csv(avg_path, index=False)
print('Saved average radial spectrum:', avg_path)

param_summary = (
    fit_df
    .groupby(['variant', 'model_label'], observed=False)
    .agg(
        n_slices=('time_position', 'nunique'),
        sigmasq_median=('est_sigmasq', 'median'),
        range_lat_median=('est_range_lat', 'median'),
        range_lon_median=('est_range_lon', 'median'),
        nugget_median=('est_nugget', 'median'),
        loss_median=('loss', 'median'),
        loss_mean=('loss', 'mean'),
    )
    .reset_index()
)
param_summary_path = OUT_DIR / f'{OUT_PREFIX}_plot_param_summary.csv'
round_df(param_summary).to_csv(param_summary_path, index=False, float_format=f'%.{ROUND_DECIMALS}f')
print('Saved plot parameter summary:', param_summary_path)
display(round_df(param_summary))

positive_vals = pd.concat([
    pd.to_numeric(avg_spec['data_spectrum'], errors='coerce'),
    pd.to_numeric(avg_spec['theory_spectrum_expected'], errors='coerce'),
], ignore_index=True).replace([np.inf, -np.inf], np.nan).dropna()
positive_vals = positive_vals[positive_vals > 0]
SPECTRUM_YLIM = (1e-1, 1e5) if positive_vals.empty else (
    10 ** np.floor(np.log10(float(positive_vals.min()))),
    10 ** np.ceil(np.log10(float(positive_vals.max()))),
)
print('Radial spectrum y-limits:', SPECTRUM_YLIM)

variant_order = list(VARIANTS)
n_cols = 3
n_rows = int(np.ceil(len(variant_order) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.8 * n_cols, 4.2 * n_rows), sharex=True, sharey=True)
axes = np.asarray(axes).reshape(n_rows, n_cols)

for ax, variant in zip(axes.ravel(), variant_order):
    label = VARIANTS[variant]['label']
    sub_avg = avg_spec[avg_spec['variant'] == variant].sort_values('k_mid')
    sub_all = plot_spec[plot_spec['variant'] == variant]
    psub = param_summary[param_summary['variant'] == variant]

    for _, hs in sub_all.groupby('time_position'):
        hs = hs.sort_values('k_mid')
        ax.plot(hs['k_mid'], hs['data_spectrum'], color='0.65', alpha=0.32, linewidth=0.75, zorder=1)

    ax.plot(sub_avg['k_mid'], sub_avg['data_spectrum'], color='black', linewidth=2.25, label='data residual spectrum', zorder=5)
    ax.plot(sub_avg['k_mid'], sub_avg['theory_spectrum_expected'], color='tab:red', linewidth=2.1, linestyle='--', label='expected periodogram', zorder=6)
    if not sub_avg.empty:
        k_cut = float(sub_avg['data_k_max'].mean())
        ax.axvline(k_cut, color='0.45', linewidth=1.0, linestyle=':', alpha=0.85, zorder=2)

    ratio = sub_avg[['k_mid', 'ratio_I_over_EI']].replace([np.inf, -np.inf], np.nan).dropna()
    ratio = ratio[(ratio['k_mid'] > 0) & (ratio['ratio_I_over_EI'] > 0)]
    if not ratio.empty:
        ratio_ax = ax.twinx()
        ratio_ax.plot(ratio['k_mid'], ratio['ratio_I_over_EI'], color='tab:blue', linewidth=1.4, linestyle=':', alpha=0.95, label='I / E[I]', zorder=7)
        ratio_ax.axhline(1.0, color='tab:blue', linewidth=0.8, alpha=0.28)
        ratio_ax.set_yscale('log')
        vals = ratio['ratio_I_over_EI'].to_numpy(dtype=float)
        lo = max(1e-3, min(0.5, float(np.nanmin(vals)) / 1.2))
        hi = min(1e3, max(2.0, float(np.nanmax(vals)) * 1.2))
        if np.isfinite(lo) and np.isfinite(hi) and lo < hi:
            ratio_ax.set_ylim(lo, hi)
        ratio_ax.tick_params(axis='y', colors='tab:blue', labelsize=8)
        ratio_ax.spines['right'].set_color('tab:blue')
        ratio_ax.set_ylabel('I / E[I]', color='tab:blue', fontsize=8)
        ratio_ax.grid(False)

    ax.set_title(label + ' | x1', fontsize=11)
    ax.set_yscale('log')
    ax.set_ylim(*SPECTRUM_YLIM)
    ax.set_xlim(0, FULL_K_PLOT_MAX)
    ax.grid(alpha=0.2)
    ax.set_xlabel('radial frequency on full-grid scale')
    ax.set_ylabel('spectrum')

    if not psub.empty:
        p = psub.iloc[0]
        param_text = (
            f"median over {int(p.n_slices)} slices\n"
            f"sigma^2={p.sigmasq_median:.3g}\n"
            f"range_lat={p.range_lat_median:.3g}\n"
            f"range_lon={p.range_lon_median:.3g}\n"
            f"nugget={p.nugget_median:.3g}\n"
            f"loss={p.loss_median:.3g}"
        )
        ax.text(0.98, 0.96, param_text, transform=ax.transAxes, ha='right', va='top', fontsize=8,
                bbox=dict(facecolor='white', edgecolor='0.85', alpha=0.78))

    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(handles, labels, fontsize=8, loc='lower left')

for ax in axes.ravel()[len(variant_order):]:
    ax.axis('off')

fig.suptitle(
    f'July {YEAR}: x1 radial residual spectrum vs fitted expected periodogram, nugget0 vs nugget free ({len(selected_time_df)} slices)',
    y=0.995,
    fontsize=14,
)
fig.tight_layout(rect=[0, 0, 1, 0.97])
plot_path = OUT_DIR / f'{OUT_PREFIX}_radial_expected_periodogram_x1_avg_3days.png'
fig.savefig(plot_path, dpi=220, bbox_inches='tight')
print('Saved x1 average radial diagnostic plot:', plot_path)
plt.show()


In [ ]:
# Frequency-band ratio tables.
# Saves full k-bin ratios, representative k-bin ratios, and coarse low/mid/high band summaries.
if 'avg_spec' not in globals():
    avg_spec = pd.read_csv(OUT_DIR / f'{OUT_PREFIX}_radial_spectrum_avg_by_variant.csv')
if 'spectral_df' not in globals():
    spectral_df = pd.read_csv(OUT_DIR / f'{OUT_PREFIX}_radial_spectrum.csv')

avg_spec = avg_spec.replace([np.inf, -np.inf], np.nan).copy()
variant_order_for_table = [v for v in VARIANTS if v in set(avg_spec['variant'].astype(str))]
label_by_variant = dict(avg_spec.drop_duplicates('variant')[['variant', 'model_label']].values)
ordered_model_labels = [label_by_variant.get(v, v) for v in variant_order_for_table if label_by_variant.get(v, v) in set(avg_spec['model_label'])]

ratio_pivot_all = (
    avg_spec
    .pivot_table(index=['k_bin', 'k_mid'], columns='model_label', values='ratio_I_over_EI', aggfunc='first')
    .reset_index()
    .sort_values(['k_bin', 'k_mid'])
)
ratio_pivot_all = ratio_pivot_all[['k_bin', 'k_mid'] + ordered_model_labels]
ratio_pivot_all_path = OUT_DIR / f'{OUT_PREFIX}_all_frequency_ratio_pivot.csv'
round_df(ratio_pivot_all).to_csv(ratio_pivot_all_path, index=False, float_format=f'%.{ROUND_DECIMALS}f')
print('Saved all-frequency ratio pivot:', ratio_pivot_all_path)

all_bins = sorted(pd.to_numeric(ratio_pivot_all['k_bin'], errors='coerce').dropna().astype(int).unique())
selected_bins = sorted(set([b for b in all_bins if b < 5] + [b for b in all_bins if b % 5 == 0] + [max(all_bins)]))
ratio_pivot_selected = ratio_pivot_all[ratio_pivot_all['k_bin'].isin(selected_bins)].copy()
ratio_pivot_selected_path = OUT_DIR / f'{OUT_PREFIX}_selected_frequency_ratio_pivot.csv'
round_df(ratio_pivot_selected).to_csv(ratio_pivot_selected_path, index=False, float_format=f'%.{ROUND_DECIMALS}f')
print('Saved selected-frequency ratio pivot:', ratio_pivot_selected_path)
display(round_df(ratio_pivot_selected))

band_defs = [
    ('ultra_low', 0, 4),
    ('low_mid', 5, 14),
    ('mid', 15, 29),
    ('high', 30, 49),
    ('very_high', 50, max(all_bins)),
]
band_rows = []
for band_name, lo, hi in band_defs:
    sub_band = avg_spec[
        (pd.to_numeric(avg_spec['k_bin'], errors='coerce') >= lo)
        & (pd.to_numeric(avg_spec['k_bin'], errors='coerce') <= hi)
    ].copy()
    if sub_band.empty:
        continue
    for variant in variant_order_for_table:
        sv = sub_band[sub_band['variant'].astype(str) == str(variant)].copy()
        if sv.empty:
            continue
        vals = pd.to_numeric(sv['ratio_I_over_EI'], errors='coerce').replace([np.inf, -np.inf], np.nan).dropna()
        band_rows.append({
            'frequency_band': band_name,
            'k_bin_start': int(lo),
            'k_bin_end': int(hi),
            'k_mid_min': float(pd.to_numeric(sv['k_mid'], errors='coerce').min()),
            'k_mid_max': float(pd.to_numeric(sv['k_mid'], errors='coerce').max()),
            'variant': variant,
            'model_label': label_by_variant.get(variant, variant),
            'n_bins': int(vals.shape[0]),
            'ratio_mean': float(vals.mean()) if len(vals) else np.nan,
            'ratio_median': float(vals.median()) if len(vals) else np.nan,
            'ratio_min': float(vals.min()) if len(vals) else np.nan,
            'ratio_max': float(vals.max()) if len(vals) else np.nan,
        })

band_ratio_summary = pd.DataFrame(band_rows)
band_ratio_summary_path = OUT_DIR / f'{OUT_PREFIX}_frequency_band_ratio_summary.csv'
round_df(band_ratio_summary).to_csv(band_ratio_summary_path, index=False, float_format=f'%.{ROUND_DECIMALS}f')
print('Saved frequency-band ratio summary:', band_ratio_summary_path)

band_ratio_mean_pivot = (
    band_ratio_summary
    .pivot_table(index=['frequency_band', 'k_mid_min', 'k_mid_max'], columns='model_label', values='ratio_mean', aggfunc='first')
    .reset_index()
)
band_order = {name: i for i, (name, _, _) in enumerate(band_defs)}
band_ratio_mean_pivot['_order'] = band_ratio_mean_pivot['frequency_band'].map(band_order)
band_ratio_mean_pivot = band_ratio_mean_pivot.sort_values('_order').drop(columns=['_order'])
band_ratio_mean_pivot = band_ratio_mean_pivot[['frequency_band', 'k_mid_min', 'k_mid_max'] + ordered_model_labels]
band_ratio_mean_pivot_path = OUT_DIR / f'{OUT_PREFIX}_frequency_band_ratio_mean_pivot.csv'
round_df(band_ratio_mean_pivot).to_csv(band_ratio_mean_pivot_path, index=False, float_format=f'%.{ROUND_DECIMALS}f')
print('Saved frequency-band mean ratio pivot:', band_ratio_mean_pivot_path)
display(round_df(band_ratio_mean_pivot))

band_ratio_median_pivot = (
    band_ratio_summary
    .pivot_table(index=['frequency_band', 'k_mid_min', 'k_mid_max'], columns='model_label', values='ratio_median', aggfunc='first')
    .reset_index()
)
band_ratio_median_pivot['_order'] = band_ratio_median_pivot['frequency_band'].map(band_order)
band_ratio_median_pivot = band_ratio_median_pivot.sort_values('_order').drop(columns=['_order'])
band_ratio_median_pivot = band_ratio_median_pivot[['frequency_band', 'k_mid_min', 'k_mid_max'] + ordered_model_labels]
band_ratio_median_pivot_path = OUT_DIR / f'{OUT_PREFIX}_frequency_band_ratio_median_pivot.csv'
round_df(band_ratio_median_pivot).to_csv(band_ratio_median_pivot_path, index=False, float_format=f'%.{ROUND_DECIMALS}f')
print('Saved frequency-band median ratio pivot:', band_ratio_median_pivot_path)
display(round_df(band_ratio_median_pivot))
